# HypotheSAEs Quickstart

This notebook demonstrates basic usage of HypotheSAEs on a sample of the Yelp review dataset.
By default we use `gpt-5.2` for hypothesis generation (interpreting neurons) and `gpt-5-mini` for text annotation.

Set `OPENAI_KEY_SAE` below for OpenAI-hosted requests. If you are using a local OpenAI-compatible endpoint (for example, vLLM server mode), set `OPENAI_BASE_URL`; the key is optional for non-OpenAI hosts.


In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ['OPENAI_KEY_SAE'] = '...'  # Replace with your OpenAI API key (or set from another env var)
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8001/v1'  # Optional: local OpenAI-compatible endpoint

import pandas as pd
from hypothesaes.quickstart import interpret_sae
from hypothesaes.embedding import get_local_embeddings

INTERPRETER_MODEL = "openai/gpt-oss-20b"
ANNOTATOR_MODEL = INTERPRETER_MODEL
# Optional request kwargs forwarded directly to the Responses API.
# Keep empty by default; set only what you need at call time.
# Example: INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
ANNOTATION_LLM_KWARGS = {"reasoning_effort": "low"}

N_WORKERS_ANNOTATION = 10  # Lower if hitting API rate limits

**Load data**

The dataset we're using here is a subset of 20K Yelp reviews, with 2K reviews used for validation (during SAE training). 

The target variable is the `stars` column, which is a rating between 1 and 5. We treat this as a regression task.

There are also 2K reviews used for holdout evaluation, which we'll use at the end of the notebook.

In [2]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
is_vast = True
if is_vast:
    train_path = os.path.join(prefix, 'data', 'vast_train.csv')
    val_path = os.path.join(prefix, 'data', 'vast_dev.csv')
    DOC_FIELD = 'post'
    TOPIC_FIELD = 'new_topic'
else:
    train_path = os.path.join(prefix, 'data', 'raw_train_all_onecol.csv')
    train_path = os.path.join(prefix, 'data', 'raw_val_all_onecol.csv')
    DOC_FIELD = "Ori Text"
    TOPIC_FIELD = "Target 1"
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
all_train_texts = list(train_df[DOC_FIELD])
all_train_topics = list(train_df[TOPIC_FIELD])
unique_train_texts  = list(train_df[DOC_FIELD].unique())
unique_train_topics = list(train_df[TOPIC_FIELD].unique())


In [3]:
all_val_texts  = list(val_df[DOC_FIELD])
all_val_topics = list(val_df[TOPIC_FIELD])
noleak_val_df = val_df[~(
    val_df[DOC_FIELD].map(set(unique_train_texts).__contains__) |
    val_df[TOPIC_FIELD].map(set(unique_train_topics).__contains__)
)]
unique_val_texts  = list(noleak_val_df[DOC_FIELD].unique())
unique_val_topics = list(noleak_val_df[TOPIC_FIELD].unique())

In [4]:
embed_func = lambda texts: get_local_embeddings(texts, return_stacked=True)
unique_train_text_embeds  = embed_func(unique_train_texts)
unique_val_text_embeds    = embed_func(unique_val_texts)
unique_train_topic_embeds = embed_func(unique_train_topics)
unique_val_topic_embeds   = embed_func(unique_val_topics)

all_train_text_embeds = embed_func(all_train_texts)
all_val_text_embeds = embed_func(all_val_texts)
all_train_topic_embeds = embed_func(all_train_topics)
all_val_topic_embeds = embed_func(all_val_topics)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


... (22 outputs left out)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/17 [00:00<?, ?it/s]

In [5]:
import importlib
import hypothesaes.sae
importlib.reload(hypothesaes.sae)
from hypothesaes.sae import SparseAutoencoder
import torch

use_abs_topk = False
shared_model_kwargs = {
    "m_total_neurons": 256,
    "k_active_neurons": 8,
    "prefix_lengths": [32, 256],
    "use_abs_topk": use_abs_topk,
}
shared_fit_kwargs = {
    "n_epochs": 300,
}
def my_train(train_embeddings, val_embeddings):
    sae = SparseAutoencoder(
        input_dim=train_embeddings.shape[1],
        **shared_model_kwargs
    )
    train_embeddings = torch.tensor(train_embeddings, dtype=torch.float)
    val_embeddings = torch.tensor(val_embeddings, dtype=torch.float)
    sae.fit(X_train=train_embeddings, X_val=val_embeddings, **shared_fit_kwargs)
    return sae
text_sae = my_train(unique_train_text_embeds, unique_val_text_embeds)
topic_sae = my_train(unique_train_topic_embeds, unique_val_topic_embeds)

  0%|          | 0/300 [00:00<?, ?it/s]

Early stopping triggered after 113 epochs


  0%|          | 0/300 [00:00<?, ?it/s]

Early stopping triggered after 111 epochs


In [25]:
import numpy as np
def get_labels(df):
    return np.array(list(map(int, df['label'])))
train_labels = get_labels(train_df)
val_labels = get_labels(val_df)

In [6]:
def mono_acts(text_embeds, topic_embeds):
    return text_sae.get_activations(text_embeds), topic_sae.get_activations(topic_embeds)
mono_train_text, mono_train_topic = mono_acts(all_train_text_embeds, all_train_topic_embeds)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
import numpy as np
from collections import Counter, defaultdict
from functools import reduce
def feature_counts(features):
    assert len(features.shape) == 2
    indices = np.where(features)
    def accum_func(accum, ind_pair):
        accum[int(ind_pair[0])].append(int(ind_pair[1]))
        return accum
    sample_groups = reduce(accum_func, zip(*indices), defaultdict(list))
    feature_counts = Counter(map(tuple, sample_groups.values()))
    return feature_counts
text_feature_counts = feature_counts(text_sae.get_activations(
    unique_train_text_embeds
))
topic_feature_counts = feature_counts(topic_sae.get_activations(
    unique_train_topic_embeds
))

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def signed_min(A, B):
    return np.sign(A) * np.sign(B) * np.minimum(np.abs(A), np.abs(B))
def conj_acts(A, B, conj=None):
    if conj is None:
        conj = signed_min if use_abs_topk else np.minimum
    n_samples = A.shape[0]
    conj_acts = conj(np.expand_dims(A, 2), np.expand_dims(B, 1))
    flattened = np.reshape(conj_acts, (n_samples, -1))
    return flattened
mono_val_text, mono_val_topic = mono_acts(all_val_text_embeds, all_val_topic_embeds)
pairwise_train = conj_acts(mono_train_text, mono_train_topic)
pairwise_val   = conj_acts(mono_val_text, mono_val_topic)


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
def concat_feats(*args):
    return np.concat([*args], axis=1)
# train_acts = concat_feats(mono_train_text, mono_train_topic, pairwise_train)
# val_acts = concat_feats(mono_val_text, mono_val_topic, pairwise_val)
train_acts = pairwise_train
val_acts = pairwise_val

In [58]:
def feature_density(samples):
    return np.sum(np.abs(samples) > 1e-6, axis=1) / samples.shape[1]

In [28]:
from sklearn.utils import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.sort(np.unique(train_labels)), y=train_labels)
train_and_val_labels = np.concat([train_labels, val_labels])
train_and_val_weights = class_weights[train_and_val_labels]

In [64]:
from cuml.linear_model import LogisticRegression
# from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.metrics import precision_recall_fscore_support
split = PredefinedSplit(test_fold=np.concat([np.full_like(train_labels, -1), np.zeros_like(val_labels)]))
param_grid = {
    "l1_ratio": [0.999],
}
cv = GridSearchCV(
    estimator=LogisticRegression(solver='qn'),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=split,
)
fitted_clf = cv.fit(
    np.concat([train_acts, val_acts]),
    train_and_val_labels,
    sample_weight=train_and_val_weights
)
val_preds = fitted_clf.predict(val_acts)
res = precision_recall_fscore_support(val_labels, val_preds)
res

(array([0.57632743, 0.56319515, 0.50887574]),
 array([0.76169591, 0.81791483, 0.12338594]),
 array([0.65617128, 0.66706587, 0.19861432]),
 array([684, 681, 697]))

In [65]:
best_est = fitted_clf.best_estimator_
est_coefs = best_est.coef_
est_coefs

array([[ 0.21469055, -0.0807718 ,  0.5760087 , ..., -0.03810808,
         0.06579751,  0.        ],
       [ 0.31177726,  0.06555404, -0.7844605 , ...,  0.06014803,
        -0.06130391,  0.        ],
       [-0.5264726 ,  0.01521843,  0.20844528, ..., -0.02204026,
        -0.00449333,  0.        ]], shape=(3, 65536), dtype=float32)

In [58]:
import plotly.express as px

In [69]:
px.histogram(est_coefs[0])

In [57]:
np.sum(np.abs(est_coefs) > 1e-1, axis=1)

array([15097, 15128,  9256])

In [39]:
inds_a, = np.where(est_coefs[0])
inds_b, = np.where(est_coefs[1])
inds_c, = np.where(est_coefs[2])

In [44]:
metric_by_class = np.stack(res[:3], axis=0)
macro_avgs = np.mean(metric_by_class, axis=1)
macro_avgs

array([0.54982776, 0.56816412, 0.50721803])

In [10]:
topic_task_instructions = """All of the below samples are topics discussed on online fora.
Features should be more general than the topics themselves."""
# Interpret random neurons
results_topics = interpret_sae(
    texts=unique_train_topics,
    embeddings=unique_train_topic_embeds,
    # texts=all_train_topics,
    # embeddings=all_train_topic_embeds,
    sae=topic_sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=topic_task_instructions,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (5015, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 156 (1.5% active): mentions the word 'politics'

Top activating examples:
1. politics
2. political
3. political thought
----------------------------------------------------------------------------------------------------

Neuron 49 (1.6% active): mentions the word 'mexican'

Top activating examples:
1. mexican
2. mexico
3. latino
----------------------------------------------------------------------------------------------------

Neuron 163 (2.4% active): mentions alliance

Top activating examples:
1. working together
2. allying with enemies in war
3. countries working together on climate change issues
----------------------------------------------------------------------------------------------------

Neuron 152 (2.3% active): mentions disaster

Top activating examples:
1. disaster
2. housing market crash
3. flag burning
----------------------------------------------------------------------------------------------------

Neuron 144 (1.9% active): mentions law

Top activating e

In [11]:
text_task_instructions = """All of the below samples are posts from an online forum.
Features should describe a specific aspect of the post (especially what topics it discusses)."""
# Interpret random neurons
results = interpret_sae(
    texts=unique_train_texts,
    embeddings=unique_train_text_embeds,
    sae=text_sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=text_task_instructions,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (1845, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 212 (3.4% active): mentions Coursera

Top activating examples:
1. Innovation, a concrete concept, was once linked to Progress, an abstract concept. This is no longer the case. We live in a world in which the main goal of innovators is to disrupt our lives by forcing us to by new stuff. As Edward Abbey said, "Growth for the sake of growth is the ideology of a cancer cell."
2. The unacknowledged advantage of privileging 'starchitects'' over the local talents is that the first come with a set of standards, a rigorous vocabulary and a name that prevail over the compromises usually inherent in a developer-driven culture (cost cuts, value-engineering, the client's 'taste', etc. - whatever could damage a project from the design phase to the actual built outcome). If one looks objectively, disregarding personal taste, whether it is a Gehry, a Nouvel or Isozaki building here in New York - they all stand out as impeccably executed architecture. Most of them relate to their context (see t